# Multi-Modal Product Classifier


## Kitabxana

In [ ]:
!pip install torch torchvision transformers scikit-learn pandas pillow tqdm -q

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

# GPU varsa, yoxsa CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
import kagglehub

import pandas as pd
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-dataset")

import os
files = os.listdir(path)
print(files)

In [ ]:
base_path = os.path.join(path, "fashion-dataset")

print(os.listdir(base_path))

In [ ]:
data = base_path + "/styles.csv"

In [ ]:
df = pd.read_csv(data, on_bad_lines="skip")
df.head()

In [ ]:
img_data = base_path + "/images/"
df['image_path'] = df['id'].astype(str).apply(
    lambda x: img_data + x + ".jpg"
)

In [ ]:
df['image_path'].unique()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
plt.pie(df['gender'].value_counts(), labels=df['gender'].value_counts().index, autopct='%1.1f%%');
df['gender'].value_counts()

In [ ]:
df['masterCategory'].value_counts()

In [ ]:
ax = sns.countplot(data=df, x='masterCategory', hue='gender',palette='dark')
for i in ax.containers:
  ax.bar_label(i)

plt.xticks(rotation=45);

In [ ]:
plt.figure(figsize=(12, 6))
plt.pie(df['season'].value_counts(), labels=df['season'].value_counts().index, autopct='%1.1f%%');

df['season'].value_counts()

In [ ]:
sns.histplot(data=df, x='year');

In [ ]:
df['usage'].value_counts()

In [ ]:
ax = sns.countplot(data=df, x='usage')
for i in ax.containers:
  ax.bar_label(i)

plt.xticks(rotation=45);

In [ ]:
def build_text(row):
    parts = [
        row.get('gender', ''),
        row.get('masterCategory', ''),
        row.get('subCategory', ''),
        row.get('articleType', ''),
        row.get('baseColour', ''),
        row.get('season', '')
    ]
    return " ".join([str(p) for p in parts if pd.notna(p)])

df['text'] = df.apply(build_text, axis=1)

In [ ]:
df.head(3)

In [ ]:
top_classes = df['masterCategory'].value_counts().nlargest(TOP_N_CLASSES).index
df = df[df['masterCategory'].isin(top_classes)].reset_index(drop=True)

In [ ]:
if MAX_SAMPLES:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
df.head(2)

In [ ]:
NUM_CLASSES

## Dataset clasterləşdirmək


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

img_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class ProductDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Şəkil
        try:
            img = Image.open(row['img_path']).convert('RGB')
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        img = img_transform(img)

        # Mətn
        enc = tokenizer(
            row['text'],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)
        return img, input_ids, attention_mask, label

In [ ]:
target_col = "masterCategory"

In [ ]:
# Hiperparametrlər
img_size    = 64
max_len     = 16
batch_size  = 32
epoch      = 3
lr          = 1e-5
max_samples = 5000
top_n_classes = 10

## Train Val ayırmaq, DataLoader yaratmaq

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['masterCategory'])
NUM_CLASSES = len(le.classes_)

In [ ]:
df_model = df[["image_path", "text", "label"]].copy()

df_model = df_model.dropna(subset=["image_path", "text", "label"])
df_model = df_model[df_model["text"].str.strip() != ""]

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_model,
    test_size=0.2,
    stratify=df_model['label'],
    random_state=42
)

# index reset
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# balans yoxlama
print("Train label distribution:")
print(train_df['label'].value_counts(normalize=True))

print("\n Val label distribution:")
print(val_df['label'].value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df_model, test_size=0.2, random_state=42, stratify=df_model['label'])

train_loader = DataLoader(ProductDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(ProductDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
val_df.head()

## ResNet-18 / DistilBERT / Fusion / Classifier

In [ ]:
'''
The model has two branches. ResNet extracts 512 features from the image. DistilBERT extracts 768
features from the text using the CLS token. These two features are concatenated into 1280
dimensions. The classifier reduces 1280 to 256, passes through ReLU and Dropout, then outputs
the number of classes. Most of DistilBERT is frozen; only the last 2 layers are trainable. All of
ResNet is trainable. This model is built for 6 categories.
'''

In [ ]:
class MultiModalModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # ResNet-18
        resnet = models.resnet18(weights='DEFAULT')
        resnet.fc = nn.Identity()           # son layeri sil, feature al
        self.img_encoder = resnet           # çıxış: 512

        # DistilBERT
        self.text_encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        # Yalnız son 2 layeri train edir qalanı donur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for param in self.text_encoder.transformer.layer[-2:].parameters():
            param.requires_grad = True

        # Fusion  Classifier
        # 512 (resnet) + 768 (distilbert) = 1280
        self.classifier = nn.Sequential(
            nn.Linear(512 + 768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, img, input_ids, attention_mask):
        # Şəkil feature-ları
        img_feat = self.img_encoder(img)

        # CLS token
        text_out = self.text_encoder(input_ids=input_ids,
                                     attention_mask=attention_mask)
        text_feat = text_out.last_hidden_state[:, 0, :]


        combined = torch.cat([img_feat, text_feat], dim=1)
        return self.classifier(combined)


model = MultiModalModel(NUM_CLASSES).to(DEVICE)
print('Model hazırdır!')
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parametr sayı: {total_params:,}')

## Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for batch in tqdm(loader, leave=False):

            img        = batch['image'].to(DEVICE)
            input_ids  = batch['input_ids'].to(DEVICE)
            attn_mask  = batch['attention_mask'].to(DEVICE)
            labels     = batch['label'].to(DEVICE)

            logits = model(img, input_ids, attn_mask)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return avg_loss, macro_f1

## Model Qiymətləndirmə

In [ ]:
"""
    Executes one complete pass through the dataset (either training or validation)

    Args:
        loader: DataLoader object containing batches of (image, text, mask, labels)
        train: Boolean flag - True for training mode, False for evaluation mode

    Returns:
        avg_loss: Average loss over all batches
        macro_f1: Macro-averaged F1 score across all classes

    What this function does:
        1. Sets model to train/eval mode (affects dropout, batch norm behavior)
        2. Iterates through all batches in the dataloader
        3. For each batch:
           - Moves data to CPU device
           - Passes image + text through model to get predictions (logits)
           - Calculates loss between predictions and true labels
           - If training: computes gradients and updates model weights
           - Collects predictions and labels for final F1 calculation
        4. Returns average loss and macro F1 score
    """

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for img, input_ids, attn_mask, labels in tqdm(loader, leave=False):
            img        = img.to(DEVICE)
            input_ids  = input_ids.to(DEVICE)
            attn_mask  = attn_mask.to(DEVICE)
            labels     = labels.to(DEVICE)

            logits = model(img, input_ids, attn_mask)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1

print(f'{"Epoch":<8} {"Train Loss":<12} {"Train F1":<12} {"Val Loss":<12} {"Val F1"}')
print('-' * 56)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True)
    vl_loss, vl_f1 = run_epoch(val_loader,   train=False)
    print(f'{epoch:<8} {tr_loss:<12.4f} {tr_f1:<12.4f} {vl_loss:<12.4f} {vl_f1:.4f}')

In [ ]:
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for img, input_ids, attn_mask, labels in val_loader:

        img        = img.to(DEVICE)
        input_ids  = input_ids.to(DEVICE)
        attn_mask  = attn_mask.to(DEVICE)
        labels     = labels.to(DEVICE)

        logits = model(img, input_ids, attn_mask)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


In [ ]:
macro_f1 = f1_score(all_labels, all_preds, average="macro")
print("Macro F1:", macro_f1)

In [ ]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=le.classes_,
    zero_division=0
))

macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
print("Macro F1:", macro_f1)

In [ ]:
print(set(all_labels))
print(len(le.classes_))

In [ ]:
# Modeli yadda saxlaamaq
torch.save(model.state_dict(), 'multimodal_model.pth')

In [ ]:
# all_labels

In [ ]:
# all_preds